In [ ]:
import pandas as pd
import csv
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model

from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Subtract, Dropout

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


#

In [ ]:
import re
import nltk
import spacy
import seaborn as sns
from tqdm import tqdm
from tqdm.notebook import tqdm
from nltk.corpus import stopwords


In [ ]:
# Read data from text
with open('/content/train_snli.txt') as file:
    data = file.readlines()

# prepare csv file
with open('data.csv', 'w', newline= '') as csvfile:
    filenames = ['source_txt', 'plagiarism_txt', 'label']
    writer = csv.DictWriter(csvfile, fieldnames=filenames)

    writer.writeheader()
    for line in tqdm(data):
        parts = line.strip().split('\t')
        source_txt = parts[0]
        plagiarishm_txt = parts[1]
        label = int(parts[2])

        writer.writerow({
            'source_txt' : source_txt,
            'plagiarism_txt' : plagiarishm_txt,
            'label' : label
        })
print('CSV file created successfully...')

  0%|          | 0/367373 [00:00<?, ?it/s]

CSV file created successfully...


In [ ]:
df = pd.read_csv('/content/data.csv')
df.head()

,source_txt,plagiarism_txt,label
0,A person on a horse jumps over a broken down a...,"A person is at a diner, ordering an omelette.",0
1,A person on a horse jumps over a broken down a...,"A person is outdoors, on a horse.",1
2,Children smiling and waving at camera,There are children present,1
3,Children smiling and waving at camera,The kids are frowning,0
4,A boy is jumping on skateboard in the middle o...,The boy skates down the sidewalk.,0


In [ ]:
df.isnull().sum()

,0
source_txt,0
plagiarism_txt,4
label,0


In [ ]:
df.duplicated().sum()

np.int64(454)

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
import re , string

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\n", " ", text)   # remove newlines
    text = text.translate(str.maketrans("", "", string.punctuation))   # remove punctucations
    return text

In [ ]:
# apply
df['source_txt'] = df['source_txt'].astype(str).apply(clean_text)
df['plagiarism_txt'] = df['plagiarism_txt'].astype(str).apply(clean_text)

In [ ]:
df.head()

,source_txt,plagiarism_txt,label
0,a person on a horse jumps over a broken down a...,a person is at a diner ordering an omelette,0
1,a person on a horse jumps over a broken down a...,a person is outdoors on a horse,1
2,children smiling and waving at camera,there are children present,1
3,children smiling and waving at camera,the kids are frowning,0
4,a boy is jumping on skateboard in the middle o...,the boy skates down the sidewalk,0


In [ ]:
source = df['source_txt'].tolist()
plagiarism = df['plagiarism_txt'].tolist()
label = df['label'].values

In [ ]:
tokenizer = Tokenizer(num_words= 20000 , oov_token= "<OOV>")
tokenizer.fit_on_texts(source + plagiarism)

source_seq = tokenizer.texts_to_sequences(source)
plagiarism_seq = tokenizer.texts_to_sequences(plagiarism)

mox_len = 50
source_pad = pad_sequences(source_seq , maxlen= mox_len )
plagiarism_pad = pad_sequences(plagiarism_seq , maxlen= mox_len )

In [ ]:
X_train_src , X_test_src , X_train_plag , X_test_plag , y_train , y_test = train_test_split(
    source_pad , plagiarism_pad , label , test_size= 0.2 , random_state= 42
)

In [ ]:
##siamese lstm

vab_size = len(tokenizer.word_index) + 1
embedding_dim = 128
lstm = 64

input_a = Input(shape=(mox_len,))
input_b = Input(shape=(mox_len,))

embedding = Embedding(vab_size , embedding_dim, input_length= mox_len)

embedding_a = embedding(input_a)
embedding_b = embedding(input_b)

lstm_layer = LSTM(lstm)
lstm_a = lstm_layer(embedding_a)
lstm_b = lstm_layer(embedding_b)

marged = Subtract()([lstm_a , lstm_b])
marged = Dense(64 , activation= 'relu')(marged)
marged = Dropout(0.5)(marged)
marged = Dense(1 , activation= 'sigmoid')(marged)


modal = Model(inputs=[input_a , input_b] , outputs= marged)
modal.compile(loss= 'binary_crossentropy' , optimizer= 'adam' , metrics= ['accuracy'])




/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
modal.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 50, 128)   │  3,874,432 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 64)        │     49,408 │ embedding_1[0][0… │
│                     │                   │            │ embedding_1[1][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ subtract (Subtract) │ (None, 64)        │          0 │ lstm_1[0][0],     │
│                     │                   │            │ lstm_1[1][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      4,160 │ subtract[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         65 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,928,065 (14.98 MB)

 Trainable params: 3,928,065 (14.98 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
#

In [ ]:
h = modal.fit(
    [X_train_src , X_train_plag] , y_train ,
    validation_split= 0.2 ,
    epochs= 1,
    batch_size= 32 ,
    callbacks= [es] ,
    verbose = 1

)

7339/7339 ━━━━━━━━━━━━━━━━━━━━ 721s 98ms/step - accuracy: 0.7956 - loss: 0.4308 - val_accuracy: 0.8334 - val_loss: 0.3720


In [ ]:
pred = modal.predict([X_test_src , X_test_plag])
pred = (pred > 0.5).astype(int)

2294/2294 ━━━━━━━━━━━━━━━━━━━━ 58s 25ms/step


In [ ]:
accuracy_score(y_test, pred)

0.8336967186307642

In [ ]:
def predict_pair(source_text, plag_text, tokenizer, max_len):
    # Convert to sequences
    src_seq = tokenizer.texts_to_sequences([source_text])
    plg_seq = tokenizer.texts_to_sequences([plag_text])

    # Pad
    src_pad = pad_sequences(src_seq, maxlen=max_len)
    plg_pad = pad_sequences(plg_seq, maxlen=max_len)

    # Predict
    pred = modal.predict([src_pad, plg_pad])[0][0]

    # Show result
    print("Prediction score:", pred)
    if pred > 0.5:
        print("✅ Likely PLAGIARIZED")
    else:
        print("❌ Likely NOT plagiarized")

    return pred

In [ ]:
src = "A woman is standing outside wearing glasses."
plg = "A woman wearing glasses is outdoors."

predict_pair(src, plg, tokenizer, mox_len)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
Prediction score: 0.97081864
✅ Likely PLAGIARIZED


np.float32(0.97081864)

In [ ]:
src = "Two kids are sleeping on a bench."
plg = "A car is driving on the road."

predict_pair(src, plg, tokenizer, mox_len)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
Prediction score: 0.0608449
❌ Likely NOT plagiarized


np.float32(0.0608449)